# 04. 세그먼트 비교 분석 (Segment Comparison Analysis)

**Dataset**: Google Merchandise Store (GA4 Public Dataset)  
**Period**: 2016-08-01 ~ 2017-08-01

## 목표
- 신규 vs 재방문 사용자 행동 차이
- 채널별 효율성 비교
- 디바이스별 핵심 지표 비교
- US vs Non-US 구매 행동 차이
- 통계 검정: 세그먼트 간 전환율 Z-test
- 고가치 사용자 프로파일링

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from google.cloud import bigquery

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120

client = bigquery.Client()
print('BigQuery 연결 성공')

## 4.1 신규 vs 재방문 사용자

In [ ]:
query_visitor = """
SELECT
  IF(visitNumber = 1, 'New', 'Returning') AS visitor_type,
  COUNT(*) AS sessions,
  COUNT(DISTINCT fullVisitorId) AS visitors,
  ROUND(AVG(totals.pageviews), 2) AS avg_pageviews,
  ROUND(AVG(totals.timeOnSite), 1) AS avg_time_on_site_sec,
  ROUND(COUNTIF(totals.bounces = 1) * 100.0 / COUNT(*), 2) AS bounce_rate_pct,
  COUNTIF(totals.transactions > 0) AS purchase_sessions,
  ROUND(COUNTIF(totals.transactions > 0) * 100.0 / COUNT(*), 4) AS conversion_rate_pct,
  ROUND(SUM(totals.totalTransactionRevenue) / 1e6, 2) AS revenue_usd,
  ROUND(
    SUM(totals.totalTransactionRevenue) / NULLIF(COUNTIF(totals.transactions > 0), 0) / 1e6, 2
  ) AS aov_usd
FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY visitor_type
ORDER BY visitor_type
"""

df_visitor = client.query(query_visitor).to_dataframe()
df_visitor

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#4e79a7', '#f28e2b']

# 전환율
axes[0].bar(df_visitor['visitor_type'], df_visitor['conversion_rate_pct'], color=colors)
axes[0].set_title('Conversion Rate (%)')
axes[0].set_ylabel('CVR %')
for i, v in enumerate(df_visitor['conversion_rate_pct']):
    axes[0].text(i, v + 0.02, f'{v:.2f}%', ha='center', fontweight='bold')

# 매출
axes[1].bar(df_visitor['visitor_type'], df_visitor['revenue_usd'], color=colors)
axes[1].set_title('Revenue (USD)')
axes[1].set_ylabel('Revenue ($)')
for i, v in enumerate(df_visitor['revenue_usd']):
    axes[1].text(i, v + v*0.01, f'${v:,.0f}', ha='center', fontweight='bold')

# AOV
axes[2].bar(df_visitor['visitor_type'], df_visitor['aov_usd'], color=colors)
axes[2].set_title('Average Order Value (USD)')
axes[2].set_ylabel('AOV ($)')
for i, v in enumerate(df_visitor['aov_usd']):
    axes[2].text(i, v + v*0.01, f'${v:,.0f}', ha='center', fontweight='bold')

plt.suptitle('New vs Returning Visitors', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4.2 채널별 성과 비교

In [ ]:
query_channel = """
SELECT
  channelGrouping AS channel,
  COUNT(*) AS sessions,
  ROUND(COUNTIF(totals.bounces = 1) * 100.0 / COUNT(*), 2) AS bounce_rate_pct,
  ROUND(COUNTIF(totals.transactions > 0) * 100.0 / COUNT(*), 4) AS conversion_rate_pct,
  ROUND(SUM(totals.totalTransactionRevenue) / 1e6, 2) AS revenue_usd,
  ROUND(SUM(totals.totalTransactionRevenue) / 1e6 / COUNT(*), 4) AS rps_usd
FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY channel
ORDER BY revenue_usd DESC
"""

df_channel = client.query(query_channel).to_dataframe()
df_channel

In [ ]:
# 채널 버블 차트: x=세션수, y=전환율, size=매출
fig, ax = plt.subplots(figsize=(10, 7))

# 매출 0인 채널 처리
df_plot = df_channel.copy()
df_plot['revenue_usd'] = df_plot['revenue_usd'].fillna(0)
sizes = df_plot['revenue_usd'] / df_plot['revenue_usd'].max() * 1000 + 50

scatter = ax.scatter(
    df_plot['sessions'], df_plot['conversion_rate_pct'],
    s=sizes, alpha=0.6, c=range(len(df_plot)), cmap='tab10', edgecolors='black'
)

for _, row in df_plot.iterrows():
    ax.annotate(row['channel'], (row['sessions'], row['conversion_rate_pct']),
                fontsize=9, ha='center', va='bottom',
                xytext=(0, 10), textcoords='offset points')

ax.set_xlabel('Sessions')
ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Channel Performance: Sessions vs CVR (bubble size = revenue)', fontsize=13)
plt.tight_layout()
plt.show()

## 4.3 디바이스별 핵심 지표

In [ ]:
query_device = """
SELECT
  device.deviceCategory AS device,
  COUNT(*) AS sessions,
  ROUND(AVG(totals.pageviews), 2) AS avg_pageviews,
  ROUND(AVG(totals.timeOnSite), 1) AS avg_time_sec,
  ROUND(COUNTIF(totals.bounces = 1) * 100.0 / COUNT(*), 2) AS bounce_rate_pct,
  ROUND(COUNTIF(totals.transactions > 0) * 100.0 / COUNT(*), 4) AS cvr_pct,
  ROUND(SUM(totals.totalTransactionRevenue) / 1e6, 2) AS revenue_usd,
  ROUND(
    SUM(totals.totalTransactionRevenue) / NULLIF(COUNTIF(totals.transactions > 0), 0) / 1e6, 2
  ) AS aov_usd
FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY device
ORDER BY sessions DESC
"""

df_device = client.query(query_device).to_dataframe()
df_device

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
colors = ['#4e79a7', '#f28e2b', '#76b7b2']

metrics = [
    ('avg_pageviews', 'Avg Pageviews', '{:.1f}'),
    ('bounce_rate_pct', 'Bounce Rate (%)', '{:.1f}%'),
    ('cvr_pct', 'Conversion Rate (%)', '{:.3f}%'),
    ('aov_usd', 'AOV (USD)', '${:.0f}')
]

for ax, (col, title, fmt) in zip(axes.flat, metrics):
    bars = ax.bar(df_device['device'], df_device[col], color=colors)
    ax.set_title(title)
    for bar, val in zip(bars, df_device[col]):
        if pd.notna(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + bar.get_height()*0.01,
                    fmt.format(val), ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Device Comparison: Key Metrics', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4.4 US vs Non-US

In [ ]:
query_geo = """
SELECT
  IF(geoNetwork.country = 'United States', 'US', 'Non-US') AS region,
  COUNT(*) AS sessions,
  COUNT(DISTINCT fullVisitorId) AS visitors,
  ROUND(AVG(totals.pageviews), 2) AS avg_pageviews,
  ROUND(COUNTIF(totals.bounces = 1) * 100.0 / COUNT(*), 2) AS bounce_rate_pct,
  COUNTIF(totals.transactions > 0) AS purchase_sessions,
  ROUND(COUNTIF(totals.transactions > 0) * 100.0 / COUNT(*), 4) AS cvr_pct,
  ROUND(SUM(totals.totalTransactionRevenue) / 1e6, 2) AS revenue_usd
FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY region
ORDER BY region
"""

df_geo = client.query(query_geo).to_dataframe()
df_geo

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
colors = ['#4e79a7', '#e15759']

# 세션 비중
axes[0].pie(df_geo['sessions'], labels=df_geo['region'], autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[0].set_title('Sessions Share')

# 전환율
axes[1].bar(df_geo['region'], df_geo['cvr_pct'], color=colors)
axes[1].set_title('Conversion Rate (%)')
for i, v in enumerate(df_geo['cvr_pct']):
    axes[1].text(i, v + 0.01, f'{v:.3f}%', ha='center', fontweight='bold')

# 매출 비중
axes[2].pie(df_geo['revenue_usd'], labels=df_geo['region'], autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[2].set_title('Revenue Share')

plt.suptitle('US vs Non-US Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4.5 통계 검정: 신규 vs 재방문 전환율 Z-test

In [ ]:
query_ztest = """
SELECT
  IF(visitNumber = 1, 'New', 'Returning') AS visitor_type,
  COUNT(*) AS total_sessions,
  COUNTIF(totals.transactions > 0) AS conversions
FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY visitor_type
ORDER BY visitor_type
"""

df_ztest = client.query(query_ztest).to_dataframe()
df_ztest

In [ ]:
# Two-proportion Z-test
new = df_ztest[df_ztest['visitor_type'] == 'New'].iloc[0]
ret = df_ztest[df_ztest['visitor_type'] == 'Returning'].iloc[0]

n1, x1 = int(new['total_sessions']), int(new['conversions'])
n2, x2 = int(ret['total_sessions']), int(ret['conversions'])

p1 = x1 / n1
p2 = x2 / n2
p_pool = (x1 + x2) / (n1 + n2)

se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
z_stat = (p1 - p2) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

# 효과 크기 (Cohen's h)
h = 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

# 95% 신뢰구간
se_diff = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
ci_lower = (p1 - p2) - 1.96 * se_diff
ci_upper = (p1 - p2) + 1.96 * se_diff

print('='*60)
print('Z-Test: New vs Returning Visitor Conversion Rate')
print('='*60)
print(f'H0: New와 Returning의 전환율에 차이가 없다')
print(f'H1: 유의미한 차이가 있다')
print(f'\nNew Visitors:       CVR = {p1*100:.4f}% ({x1:,} / {n1:,})')
print(f'Returning Visitors: CVR = {p2*100:.4f}% ({x2:,} / {n2:,})')
print(f'\nDifference: {(p1-p2)*100:.4f}pp')
print(f'95% CI: [{ci_lower*100:.4f}%, {ci_upper*100:.4f}%]')
print(f'\nZ-statistic: {z_stat:.4f}')
print(f'p-value: {p_value:.2e}')
print(f"Cohen's h (effect size): {h:.4f}")

if p_value < 0.05:
    print(f'\n✅ p < 0.05 → 통계적으로 유의미한 차이가 있다.')
else:
    print(f'\n❌ p >= 0.05 → 유의미한 차이를 확인할 수 없다.')

## 4.6 고가치 사용자 프로파일링

In [ ]:
query_high_value = """
WITH user_revenue AS (
  SELECT
    fullVisitorId,
    COUNT(*) AS total_sessions,
    SUM(totals.totalTransactionRevenue) / 1e6 AS total_revenue_usd,
    COUNTIF(totals.transactions > 0) AS purchase_count,
    MAX(device.deviceCategory) AS primary_device,
    MAX(channelGrouping) AS primary_channel
  FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
  GROUP BY fullVisitorId
  HAVING total_revenue_usd > 0
),
percentiles AS (
  SELECT APPROX_QUANTILES(total_revenue_usd, 10)[OFFSET(9)] AS p90_revenue
  FROM user_revenue
)
SELECT
  IF(ur.total_revenue_usd >= p.p90_revenue, 'Top 10%', 'Bottom 90%') AS segment,
  COUNT(*) AS users,
  ROUND(AVG(ur.total_sessions), 1) AS avg_sessions,
  ROUND(AVG(ur.purchase_count), 1) AS avg_purchases,
  ROUND(AVG(ur.total_revenue_usd), 2) AS avg_revenue,
  ROUND(SUM(ur.total_revenue_usd), 2) AS total_revenue
FROM user_revenue ur
CROSS JOIN percentiles p
GROUP BY segment
ORDER BY segment DESC
"""

df_hv = client.query(query_high_value).to_dataframe()
df_hv

In [ ]:
# 파레토 법칙 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#e15759', '#4e79a7']

# 사용자 수 vs 매출 기여
axes[0].pie(df_hv['users'], labels=df_hv['segment'], autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[0].set_title('User Count Share')

axes[1].pie(df_hv['total_revenue'], labels=df_hv['segment'], autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('Revenue Share')

plt.suptitle('Top 10% vs Bottom 90% Revenue Contributors', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

top10 = df_hv[df_hv['segment'] == 'Top 10%'].iloc[0]
total_rev = df_hv['total_revenue'].sum()
print(f"\nTop 10% 사용자가 전체 매출의 {top10['total_revenue']/total_rev*100:.1f}%를 기여")
print(f"Top 10% 평균 매출: ${top10['avg_revenue']:,.2f}")
print(f"Top 10% 평균 구매 횟수: {top10['avg_purchases']:.1f}회")

## Key Findings

1. **재방문 사용자 > 신규**: 재방문 사용자의 전환율, AOV 모두 신규 대비 높음
   - 재방문 유도 전략의 ROI가 높음

2. **채널 효율성 편차**: Referral, Organic의 세션당 매출이 높음
   - Social은 세션 수 대비 전환이 낮음 → 인지도 채널로 활용

3. **모바일 갭**: Desktop 대비 모바일은 이탈률 높고, 전환율과 AOV 모두 낮음
   - 모바일 결제 UX가 핵심 병목

4. **US 집중**: 매출의 대부분이 US에서 발생
   - 국제화보다 US 내 성장에 집중하는 것이 효율적

5. **파레토 법칙**: 상위 10% 구매자가 매출의 상당 부분 기여
   - VIP 프로그램, 로열티 마케팅의 ROI가 높을 것

### Action Items
- 재방문자 전환 경로 최적화 (재구매 할인, 개인화 추천)
- 모바일 결제 프로세스 간소화 (1-click checkout)
- 상위 고객 VIP 프로그램 도입